# N-Gram Language Modeling with Unknown Words

| Assignment field | Details |
|---|---|
| **Group Number** | 90 |
| **Problem Statement** | 9 - N-gram Language Modeling with Unknown Words |
| **Dataset** | NLTK Gutenberg Corpus |
| **Student Name(s)** |  |
| **Course/Subject** |  |

## 1. Introduction

A **language model** assigns probabilities to sequences of words and can use them to predict likely words. An **N-gram** is a sequence of *N* consecutive tokens. A **trigram** model uses the previous two words to estimate the next word:

$$P(w_i \mid w_{i-2}, w_{i-1})$$

Words absent from the training vocabulary are a problem because the model has no counts for them. We use the special token `<UNK>` (unknown) to group rare training words and represent unseen test words. This gives the model a learned way to handle unfamiliar vocabulary.

## 2. Objective

This assignment will:

- build a trigram language model;
- analyze rare words and replace training words occurring fewer than 3 times with `<UNK>`;
- perform next-word and missing-word prediction;
- compare models with and without `<UNK>`;
- compare no smoothing with Laplace (add-one) smoothing; and
- evaluate accuracy, OOV rate, coverage, and perplexity.

## 3. Import libraries and reproducible setup

The setup checks for NLTK resources and downloads them only when absent. The fixed seed makes any sampling deterministic.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import random
import re
import sys

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 90
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 90)

# Keep downloaded NLTK resources in an ignored local cache when possible.
NLTK_DATA_DIR = Path.cwd() / ".nltk_data"
NLTK_DATA_DIR.mkdir(exist_ok=True)
if str(NLTK_DATA_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DATA_DIR))

required_resources = {
    "gutenberg": "corpora/gutenberg",
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab",
}
for package, resource_path in required_resources.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package, download_dir=str(NLTK_DATA_DIR), quiet=True, raise_on_error=True)

from nltk.corpus import gutenberg

print(f"Python: {sys.version.split()[0]}")
print(f"NLTK: {nltk.__version__}")
print(f"Random seed: {SEED}")

## 4. Load and inspect the NLTK Gutenberg Corpus

We first list every available file, then select one sufficiently large book. `austen-emma.txt` is suitable because it contains many sentences and repeated everyday vocabulary, while still containing rare words for the `<UNK>` experiment.

In [ ]:
available_books = gutenberg.fileids()
selected_book = "austen-emma.txt"

books_df = pd.DataFrame({"Available Gutenberg file": available_books})
display(books_df)
print(f"Number of books available: {len(available_books)}")
print(f"Selected book: {selected_book}")

In [ ]:
raw_text = gutenberg.raw(selected_book)
raw_sentences = list(gutenberg.sents(selected_book))
raw_tokens = list(gutenberg.words(selected_book))

corpus_stats_df = pd.DataFrame({
    "Statistic": ["Selected book", "Raw characters", "Raw sentences", "Tokens before preprocessing"],
    "Value": [selected_book, len(raw_text), len(raw_sentences), len(raw_tokens)],
})
display(corpus_stats_df)

## 5. Text preprocessing

Preprocessing decisions are explicit:

1. NLTK's Gutenberg sentence boundaries are retained.
2. Tokens are lowercased so `The` and `the` share counts.
3. Alphabetic words and internal apostrophes are kept; standalone punctuation and numbers are removed.
4. Two `<START>` tokens and one `<END>` token are added during trigram counting. Two starts are required because a trigram needs two preceding tokens.
5. `<UNK>` is **not** added yet; it is learned from training frequencies later.

In [ ]:
WORD_PATTERN = re.compile(r"^[A-Za-z]+(?:'[A-Za-z]+)?$")

def preprocess_sentence(tokens):
    # Lowercase and retain word-like tokens while preserving the sentence as a list.
    return [token.lower() for token in tokens if WORD_PATTERN.fullmatch(token)]

processed_sentences = [preprocess_sentence(sentence) for sentence in raw_sentences]
processed_sentences = [sentence for sentence in processed_sentences if sentence]

example_rows = []
for original, processed in zip(raw_sentences[:5], processed_sentences[:5]):
    example_rows.append({
        "Original sentence": " ".join(original[:30]),
        "After preprocessing": " ".join(processed[:30]),
    })
display(pd.DataFrame(example_rows))

## 6. Train/test data preparation

The corpus is split at sentence boundaries: the first 80% of sentences are training data and the remaining 20% are test data. This chronological, deterministic split prevents test sentences from influencing training counts or the `<UNK>` vocabulary.

In [ ]:
split_index = int(0.80 * len(processed_sentences))
train_sentences = processed_sentences[:split_index]
test_sentences = processed_sentences[split_index:]

split_df = pd.DataFrame({
    "Split": ["Training", "Testing", "Total"],
    "Sentences": [len(train_sentences), len(test_sentences), len(processed_sentences)],
    "Percentage": [100 * len(train_sentences) / len(processed_sentences),
                   100 * len(test_sentences) / len(processed_sentences), 100.0],
})
display(split_df.style.format({"Percentage": "{:.1f}%"}))

## 7. Exploratory Data Analysis

EDA is reported on the training split because training frequencies determine the vocabulary and rare-word rule. This also avoids looking at test data when designing `<UNK>`.

In [ ]:
train_tokens = [word for sentence in train_sentences for word in sentence]
test_tokens = [word for sentence in test_sentences for word in sentence]
train_frequency = Counter(train_tokens)

words_once = sum(count == 1 for count in train_frequency.values())
words_twice = sum(count == 2 for count in train_frequency.values())
rare_types = sum(count < 3 for count in train_frequency.values())
rare_token_count = sum(count for count in train_frequency.values() if count < 3)

frequency_stats_df = pd.DataFrame({
    "Statistic": ["Total training tokens", "Unique training words", "Words occurring once",
                  "Words occurring twice", "Word types occurring < 3 times",
                  "Rare-token occurrences", "Rare types as % of vocabulary"],
    "Value": [len(train_tokens), len(train_frequency), words_once, words_twice,
              rare_types, rare_token_count, 100 * rare_types / len(train_frequency)],
})
display(frequency_stats_df.style.format({"Value": lambda x: f"{x:,.2f}" if isinstance(x, float) else f"{x:,}"}))
display(pd.DataFrame(train_frequency.most_common(20), columns=["Word", "Frequency"]))

In [ ]:
top_words = pd.DataFrame(train_frequency.most_common(20), columns=["Word", "Frequency"])
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(top_words["Word"], top_words["Frequency"], color="#4C78A8")
ax.set_title("Top 20 Most Frequent Training Words")
ax.set_xlabel("Word")
ax.set_ylabel("Frequency")
ax.tick_params(axis="x", rotation=55)
plt.tight_layout()
plt.show()
print("Interpretation: Function words dominate the corpus, so they are likely predictions in many common contexts.")

In [ ]:
frequency_of_frequencies = Counter(train_frequency.values())
freq_df = pd.DataFrame(sorted(frequency_of_frequencies.items()), columns=["Word frequency", "Number of word types"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(freq_df["Word frequency"], freq_df["Number of word types"], alpha=0.65, color="#F58518")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Word-Frequency Distribution (Log-Log Scale)")
ax.set_xlabel("Frequency of a word (log scale)")
ax.set_ylabel("Number of word types (log scale)")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print("Interpretation: Most word types occur only a few times, while a small number occur very often.")

In [ ]:
rare_distribution = pd.DataFrame({
    "Training frequency": ["Once", "Twice"],
    "Number of word types": [words_once, words_twice],
})
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(rare_distribution["Training frequency"], rare_distribution["Number of word types"], color=["#E45756", "#72B7B2"])
ax.bar_label(bars, fmt="%d")
ax.set_title("Rare-Word Frequency Distribution")
ax.set_xlabel("Occurrence count")
ax.set_ylabel("Number of word types")
plt.tight_layout()
plt.show()
print("Interpretation: The large rare-word tail motivates grouping frequency-1 and frequency-2 words as <UNK>.")

## 8. Trigram model without `<UNK>`

For context $h=(w_{i-2},w_{i-1})$, the unsmoothed maximum-likelihood estimate is:

$$P(w_i\mid h)=\frac{Count(h,w_i)}{Count(h)}$$

The class below stores trigram, context, and unigram counts. Prediction is deterministic: the highest probability wins, then training frequency and alphabetical order break ties.

In [ ]:
START, END, UNK = "<START>", "<END>", "<UNK>"

class TrigramLanguageModel:
    def __init__(self, sentences, use_unk=False, rare_threshold=3):
        self.use_unk = use_unk
        self.rare_threshold = rare_threshold
        original_counts = Counter(word for sentence in sentences for word in sentence)
        self.original_vocabulary = set(original_counts)
        self.rare_words = {word for word, count in original_counts.items() if count < rare_threshold} if use_unk else set()
        self.sentences = [self.transform_sentence(sentence, training=True) for sentence in sentences]
        self.word_counts = Counter(word for sentence in self.sentences for word in sentence)
        self.vocabulary = set(self.word_counts) | {END}
        if use_unk:
            self.vocabulary.add(UNK)
        self.prediction_vocabulary = sorted(self.vocabulary - {START})
        self.trigram_counts = Counter()
        self.context_counts = Counter()
        for sentence in self.sentences:
            padded = [START, START] + sentence + [END]
            for i in range(2, len(padded)):
                context = (padded[i - 2], padded[i - 1])
                target = padded[i]
                self.trigram_counts[(context[0], context[1], target)] += 1
                self.context_counts[context] += 1

    def map_word(self, word):
        if not self.use_unk:
            return word
        return word if word in self.vocabulary else UNK

    def transform_sentence(self, sentence, training=False):
        if self.use_unk:
            if training:
                return [UNK if word in self.rare_words else word for word in sentence]
            return [self.map_word(word) for word in sentence]
        return list(sentence)

    def probability(self, word, context, smoothing=False):
        context = tuple(self.map_word(w) for w in context[-2:])
        word = self.map_word(word)
        count = self.trigram_counts[(context[0], context[1], word)]
        context_count = self.context_counts[context]
        if smoothing:
            return (count + 1) / (context_count + len(self.vocabulary))
        return count / context_count if context_count else 0.0

    def predict_next(self, context_words, smoothing=False):
        mapped = [self.map_word(w) for w in context_words]
        context = ([START, START] + mapped)[-2:]
        candidates = self.prediction_vocabulary
        if not smoothing:
            observed = [w for w in candidates if self.trigram_counts[(context[0], context[1], w)] > 0]
            if not observed:
                return None
            candidates = observed
        return min(candidates, key=lambda w: (-self.probability(w, context, smoothing), -self.word_counts[w], w))

    def sentence_perplexity(self, sentences, smoothing=False):
        log_probability_sum = 0.0
        predicted_tokens = 0
        for sentence in sentences:
            mapped = self.transform_sentence(sentence, training=False)
            padded = [START, START] + mapped + [END]
            for i in range(2, len(padded)):
                probability = self.probability(padded[i], padded[i-2:i], smoothing)
                predicted_tokens += 1
                if probability <= 0:
                    return math.inf
                log_probability_sum += math.log(probability)
        return math.exp(-log_probability_sum / predicted_tokens)

base_model = TrigramLanguageModel(train_sentences, use_unk=False)
print(f"Base vocabulary size (including <END>): {len(base_model.vocabulary):,}")
print(f"Distinct observed trigrams: {len(base_model.trigram_counts):,}")

## 9. Trigram model with `<UNK>`

Only **training frequencies** decide which words are rare. Every training word with frequency `< 3` is replaced by `<UNK>`, and test words outside the resulting training vocabulary are mapped only when passed to this model. Original test sentences remain unchanged.

- A **rare word** has a low training count (here 1 or 2).
- An **unknown word** is unavailable to the model at prediction time.
- An **out-of-vocabulary (OOV) word** is absent from the model's training vocabulary.

In [ ]:
unk_model = TrigramLanguageModel(train_sentences, use_unk=True, rare_threshold=3)
vocabulary_comparison_df = pd.DataFrame({
    "Statistic": ["Vocabulary size", "Rare word types replaced", "Training token occurrences mapped to <UNK>", "Percent of training tokens mapped"],
    "Before <UNK>": [len(base_model.vocabulary), 0, 0, 0.0],
    "After <UNK>": [len(unk_model.vocabulary), len(unk_model.rare_words), unk_model.word_counts[UNK],
                    100 * unk_model.word_counts[UNK] / len(train_tokens)],
})
display(vocabulary_comparison_df.style.format({"Before <UNK>": "{:,.2f}", "After <UNK>": "{:,.2f}"}))

fig, ax = plt.subplots(figsize=(7, 4))
values = [len(base_model.vocabulary), len(unk_model.vocabulary)]
bars = ax.bar(["Before <UNK>", "After <UNK>"], values, color=["#4C78A8", "#54A24B"])
ax.bar_label(bars, fmt="%d")
ax.set_title("Vocabulary Size Before and After <UNK>")
ax.set_xlabel("Vocabulary treatment")
ax.set_ylabel("Number of vocabulary items")
plt.tight_layout()
plt.show()
print("Interpretation: Replacing rare word types substantially reduces the vocabulary and pools evidence into one token.")

In [ ]:
oov_test_tokens = [word for word in test_tokens if word not in base_model.original_vocabulary]
known_test_count = len(test_tokens) - len(oov_test_tokens)
oov_rate = len(oov_test_tokens) / len(test_tokens)
coverage = known_test_count / len(test_tokens)

coverage_df = pd.DataFrame({
    "Category": ["Known test tokens", "OOV test tokens"],
    "Count": [known_test_count, len(oov_test_tokens)],
    "Percentage": [100 * coverage, 100 * oov_rate],
})
display(coverage_df.style.format({"Percentage": "{:.2f}%"}))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(coverage_df["Category"], coverage_df["Count"], color=["#54A24B", "#E45756"])
ax.bar_label(bars, fmt="%d")
ax.set_title("Known vs OOV Test-Token Coverage")
ax.set_xlabel("Test-token category")
ax.set_ylabel("Number of tokens")
plt.tight_layout()
plt.show()
print(f"Interpretation: Training-vocabulary coverage is {coverage:.2%}; the remaining {oov_rate:.2%} is OOV.")

## 10. No smoothing and Laplace smoothing

Unsmoothed models assign probability zero to unobserved trigrams. Laplace smoothing adds one to every possible next-word count:

$$P(w\mid h)=\frac{Count(h,w)+1}{Count(h)+V}$$

Here, $V$ is the model's prediction vocabulary size, including `<END>` and (for the unknown-word model) `<UNK>`. Add-one smoothing makes in-vocabulary events non-zero. However, a word outside the no-`<UNK>` vocabulary remains unsupported, so its test probability is still zero. For an entirely unseen context, all candidates tie; the deterministic tie-breaker chooses the most frequent training token.

In [ ]:
unknown_examples = []
for sentence in test_sentences:
    mapped = unk_model.transform_sentence(sentence)
    if sentence != mapped:
        unknown_examples.append({"Original": " ".join(sentence[:16]), "Mapped for <UNK> model": " ".join(mapped[:16])})
    if len(unknown_examples) == 5:
        break
display(pd.DataFrame(unknown_examples))